# Spark DataFrames Exercise

**Practicing PySpark DataFrame operations with Walmart (WMT) stock data.**

## Project Overview

This exercise-based notebook builds PySpark DataFrame skills using Walmart stock data. It covers reading data, schema inspection, aggregations, window functions, and SQL queries — essential skills for big data processing.

## Learning Objectives

1. Load and inspect CSV data in PySpark
2. Perform column operations and transformations
3. Use groupBy, aggregation, and ordering
4. Apply SQL queries to Spark DataFrames
5. Calculate financial metrics (returns, averages)

## Business / Research Problem

Financial analysts need to process large volumes of stock data efficiently. PySpark enables:
- Distributed processing of historical data
- Quick aggregation across time periods
- SQL-like querying for ad-hoc analysis

**Key question:** *How do we use PySpark to answer common financial analysis questions about stock data?*

## Why This Analysis Matters

Stock data analysis is a common use case for big data tools. Learning PySpark with financial data prepares you for quantitative finance and data engineering roles.

## Dataset Overview

Walmart (WMT) stock data with columns: `Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume`.

**File:** `WMT.csv` in the project directory

## Dataset Source & License Notes

- **Source:** Yahoo Finance historical data
- **License:** Public financial data
- **File:** Local `WMT.csv`

## Environment Setup

In [ ]:
!pip install pyspark pandas numpy matplotlib -q

## Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, max, min, count, stddev, year, month
from pyspark.sql.functions import format_number, corr, dayofyear, date_format
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Imports loaded.')

## Configuration / Constants

In [ ]:
DATA_FILE = 'WMT.csv'
APP_NAME = 'Spark_WMT_Exercise'

## Dataset Download / Loading

In [ ]:
spark = SparkSession.builder.appName(APP_NAME).getOrCreate()
print(f'Spark version: {spark.version}')

df = spark.read.csv(DATA_FILE, header=True, inferSchema=True)
print(f'Loaded {df.count()} rows')
df.show(5)

## Data Validation Checks

In [ ]:
df.printSchema()
print(f'\nColumns: {df.columns}')
df.describe().show()

## Data Cleaning

In [ ]:
# Check nulls
from pyspark.sql.functions import sum as spark_sum
df.select([spark_sum(col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()
df = df.dropna()
print(f'Rows after cleaning: {df.count()}')

## Exploratory Data Analysis

### Exercise 1: What are the column names?

In [ ]:
print(f'Column names: {df.columns}')

### Exercise 2: What does the first 5 rows look like?

In [ ]:
df.show(5)

### Exercise 3: Describe the DataFrame

In [ ]:
df.describe().show()

## Univariate Analysis

### Exercise 4: Create a new column HV_Ratio (High/Volume)

In [ ]:
df = df.withColumn('HV_Ratio', col('High') / col('Volume'))
df.select('Date', 'High', 'Volume', format_number('HV_Ratio', 8).alias('HV_Ratio')).show(5)

## Bivariate / Multivariate Analysis

### Exercise 5: What day had the Peak High in Price?

In [ ]:
df.orderBy(col('High').desc()).select('Date', 'High').show(1)

### Exercise 6: What is the mean of the Close column?

In [ ]:
df.select(format_number(mean('Close'), 2).alias('Mean_Close')).show()

## Feature-Specific Insights

### Exercise 7: Max and Min of Volume

In [ ]:
df.select(max('Volume').alias('Max_Volume'), min('Volume').alias('Min_Volume')).show()

### Exercise 8: How many days was the Close lower than 60?

In [ ]:
count_below = df.filter(col('Close') < 60).count()
print(f'Days with Close < 60: {count_below}')

## Statistical Checks / Hypothesis Testing

### Exercise 9: Percentage of time High > 80

In [ ]:
total = df.count()
above_80 = df.filter(col('High') > 80).count()
print(f'Days High > 80: {above_80}/{total} = {above_80/total*100:.2f}%')

### Exercise 10: Correlation between High and Volume

In [ ]:
df.select(corr('High', 'Volume').alias('High_Volume_Correlation')).show()

### Exercise 11: Max High per Year

In [ ]:
df_year = df.withColumn('Year', year(col('Date')))
df_year.groupBy('Year').agg(max('High').alias('Max_High')).orderBy('Year').show(20)

## Visual Analysis

### Exercise 12: Average Close by Month

In [ ]:
df_month = df.withColumn('Month', month(col('Date')))
monthly_avg = df_month.groupBy('Month').agg(
    format_number(mean('Close'), 2).alias('Avg_Close')
).orderBy('Month')
monthly_avg.show(12)

In [ ]:
# Visualize
pdf = df.toPandas()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(pdf)), pdf['Close'], alpha=0.7, color='steelblue')
ax.set_title('WMT Close Price Over Time')
ax.set_ylabel('Close Price')
plt.tight_layout()
plt.show()

## Key Findings

1. PySpark DataFrame operations closely mirror SQL and pandas patterns
2. Column operations use `withColumn` and `col()` for transformations
3. `groupBy().agg()` is the core summarization pattern
4. Date functions (`year`, `month`) enable time-series aggregation
5. `format_number` improves readability of numeric output

## Limitations

1. Small dataset — PySpark overhead not justified
2. Local mode only — no distributed benefits
3. No streaming or real-time data

## Recommendations / Next Steps

1. Try with multiple stocks joined together
2. Implement window functions for moving averages
3. Scale to a larger dataset to see PySpark benefits

### How to Extend This Analysis
- Add technical indicators (RSI, MACD) using window functions
- Join with earnings data for event studies
- Build a PySpark MLlib model for price prediction

## Common Mistakes

1. **Forgetting `col()` for column references** in expressions
2. **Using `collect()` on large DataFrames** — use `show()` or `take()` instead
3. **Not caching intermediate results** when reusing them
4. **Mixing Python and Spark operations** — keep transformation logic in Spark
5. **Not checking schema** before operations — inferSchema may get types wrong

## Mini Challenge / Exercises

1. Calculate the daily return percentage: (Close - Open) / Open * 100
2. Find the 5 highest volume days. What happened around those dates?
3. Calculate the 50-day moving average using window functions
4. What is the average volume by day of week?
5. Create a Spark SQL query that finds all days where Close > Open AND Volume > average volume

In [ ]:
# Space for exercises

In [ ]:
spark.stop()
print('SparkSession stopped.')

## Final Summary / Key Takeaways

- **PySpark DataFrames** provide a distributed, SQL-like API for large data
- **Key operations:** withColumn, select, filter, groupBy, agg, orderBy
- **Date functions** enable powerful time-series analysis
- **Spark SQL** provides an alternative query interface
- **Practice with financial data** builds both Spark and domain skills

These exercises cover the core operations you'll use in 90% of PySpark work.